In [9]:
# IIP data

import pandas as pd
import holidays

# 1. Load your uploaded IIP data
df = pd.read_csv('iip_3.csv')

# Create a clean datetime column for the 1st of each period month
df['period_start'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'] + '-01')

# 2. Create the master timeline spine from Jan 2010 to May 2026
spine = pd.date_range(start='2010-01-01', end='2026-05-01', freq='MS')
master_df = pd.DataFrame({'period_start': spine})

# Setup Indian Holiday Calendar for accurate business day calculations
in_holidays = holidays.India(years=range(2010, 2027), subdiv='MH')

def get_last_bday(dt):
    eom = dt + pd.offsets.MonthEnd(0)
    # Roll backward if it lands on a weekend or public holiday
    while eom.weekday() > 4 or eom in in_holidays:
        eom -= pd.Timedelta(days=1)
    return eom

def get_release_date(dt):
    eom = dt + pd.offsets.MonthEnd(0)
    release = eom + pd.Timedelta(days=42) 
    # Roll forward if the release date lands on a weekend or holiday
    while release.weekday() > 4 or release in in_holidays:
        release += pd.Timedelta(days=1)
    return release

# Apply our business day logic to the timeline
master_df['last_business_day'] = master_df['period_start'].apply(get_last_bday)
master_df['release_date'] = master_df['period_start'].apply(get_release_date)

# 3. Merge the YoY Growth Rate ('growth_rate' column) and raw level ('index' column)
df_merged = pd.merge(master_df, df[['period_start', 'growth_rate', 'index']], on='period_start', how='left')

# Ensure the dataframe is perfectly chronological before processing
df_merged = df_merged.sort_values('period_start').reset_index(drop=True)

# 4. --- STEP ADDED: Forward fill missing values within the timeline bounds ---
df_merged['growth_rate'] = df_merged['growth_rate'].ffill()
df_merged['index'] = df_merged['index'].ffill()

# 5. Calculate the 3-Month Rolling Average of the YoY Growth using the filled values
df_merged['growth_3m_avg'] = df_merged['growth_rate'].rolling(window=3).mean()

# Finalize the columns
final_df = df_merged[['last_business_day', 'release_date', 'growth_rate', 'growth_3m_avg', 'index']]
final_df = final_df.rename(columns={'index': 'IIP_Level'})

final_df.to_csv('IIP_Growth_Formatted_Spine.csv', index=False)

In [10]:
#gst collections

import pandas as pd

# 1. Load the GST CSV file, skipping original metadata title row
df_gst = pd.read_csv("All India GST Monthly Revenue.csv", skiprows=1)

month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 
             'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}

df_gst['Month_Num'] = df_gst['Month'].str.strip().map(month_map)
df_gst['date_parsed'] = pd.to_datetime(df_gst['Year'].astype(str) + '-' + df_gst['Month_Num'].astype(str) + '-01')

# Sort chronologically to correctly compute historic YoY rates
df_gst = df_gst.sort_values('date_parsed').reset_index(drop=True)

# 2. Compute YoY Growth Rate (%)
df_gst['GST_YoY_Growth'] = df_gst['Total GST Revenue (₹ Crore)'].pct_change(12) * 100

def get_first_bday_next_month(row):
    year = int(row['Year'])
    month_num = int(row['Month_Num'])
    
    if month_num == 12:
        next_month = 1
        next_year = year + 1
    else:
        next_month = month_num + 1
        next_year = year
        
    first_day = pd.Timestamp(year=next_year, month=next_month, day=1)
    # Roll forward if the 1st day of the next month lands on a weekend
    while first_day.weekday() > 4:
        first_day += pd.Timedelta(days=1)
        
    return first_day.strftime('%Y-%m-%d')

df_gst['Release Date'] = df_gst.apply(get_first_bday_next_month, axis=1)

# Isolate and align columns right after 'Month'
final_gst = df_gst[['Year-Month', 'Year', 'Month', 'Release Date', 'Total GST Revenue (₹ Crore)', 'GST_YoY_Growth']]
final_gst.to_csv("All_India_GST_Revenue_With_YoY.csv", index=False)
print("GST Data with YoY metrics exported.")

GST Data with YoY metrics exported.


In [11]:
#merchandise exports

import pandas as pd

# 1. Load raw Trade Data and skip metadata headers
df = pd.read_csv("Broad Commodity Composition of India's Merchandise Trade - Oil and Non-Oil Exports and Imports - US Dollar.csv", skiprows=5)

# Clean structural rows and handle spaces
df = df.iloc[2:].copy()
df = df.dropna(subset=['Year', 'Month'])
df['Month'] = df['Month'].str.strip()

valid_months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
df = df[df['Month'].isin(valid_months)]
month_map = {m: i+1 for i, m in enumerate(valid_months)}

# Handle Fiscal to Calendar Year adjustments
def parse_calendar_year(row):
    year_str = str(row['Year']).strip()
    month = row['Month']
    first_part = int(year_str.split('-')[0])
    if month in ['January', 'February', 'March']:
        return first_part + 1
    else:
        return first_part

df['Calendar_Year'] = df.apply(parse_calendar_year, axis=1)
df['period_start'] = pd.to_datetime(df['Calendar_Year'].astype(str) + '-' + df['Month'].map(month_map).astype(str) + '-01')

# Convert string values to numeric and compute raw metrics
df['Oil_Exp'] = df['Exports'].str.replace(',', '', regex=False).astype(float)
df['NonOil_Exp'] = df['Unnamed: 3'].str.replace(',', '', regex=False).astype(float)
df['Non-Oil Imports'] = df['Unnamed: 5'].str.replace(',', '', regex=False).astype(float)

df['Merchandise Exports'] = df['Oil_Exp'] + df['NonOil_Exp']

# 2. Generate Reference Spine (Extended back to 2009 to seed 2010 YoY windows cleanly)
spine = pd.date_range(start='2009-01-01', end='2026-05-01', freq='MS')
master_df = pd.DataFrame({'period_start': spine})

def get_last_bday(dt):
    eom = dt + pd.offsets.MonthEnd(0)
    while eom.weekday() > 4:
        eom -= pd.Timedelta(days=1)
    return eom

def get_release_date_trade(dt):
    # Last business day of the following month (30-day reporting window lag)
    following_month_start = dt + pd.offsets.MonthBegin(1)
    eom_next = following_month_start + pd.offsets.MonthEnd(0)
    while eom_next.weekday() > 4:
        eom_next -= pd.Timedelta(days=1)
    return eom_next

master_df['Month End Date'] = master_df['period_start'].apply(get_last_bday)
master_df['Release Date'] = master_df['period_start'].apply(get_release_date_trade)

# 3. Merge data onto spine and forward fill edge-gaps 
df_merged = pd.merge(master_df, df[['period_start', 'Merchandise Exports', 'Non-Oil Imports']], on='period_start', how='left')
df_merged = df_merged.sort_values('period_start').reset_index(drop=True)

df_merged['Merchandise Exports'] = df_merged['Merchandise Exports'].ffill()
df_merged['Non-Oil Imports'] = df_merged['Non-Oil Imports'].ffill()

# 4. --- NEW CALCULATIONS: YoY Changes & 3M Rolling Averages ---
df_merged['Exports_YoY_Change'] = df_merged['Merchandise Exports'].pct_change(12) * 100
df_merged['Imports_NonOil_YoY_Change'] = df_merged['Non-Oil Imports'].pct_change(12) * 100

df_merged['Exports_YoY_3MMA'] = df_merged['Exports_YoY_Change'].rolling(window=3).mean()
df_merged['Imports_NonOil_YoY_3MMA'] = df_merged['Imports_NonOil_YoY_Change'].rolling(window=3).mean()

# Slice to strictly map your target bounds: Jan 2010 to May 2026
final_trade = df_merged[(df_merged['period_start'] >= '2010-01-01') & (df_merged['period_start'] <= '2026-05-01')].copy()

# Layout alignment
final_trade = final_trade[['Month End Date', 'Release Date', 
                           'Merchandise Exports', 'Exports_YoY_Change', 'Exports_YoY_3MMA', 
                           'Non-Oil Imports', 'Imports_NonOil_YoY_Change', 'Imports_NonOil_YoY_3MMA']]

# Export cleaned configuration
final_trade.to_csv("Merchandise_Trade_Cleaned.csv", index=False)
print("Merchandise Trade Engine Processed and Exported.")

Merchandise Trade Engine Processed and Exported.


In [12]:
#services PMI

import pandas as pd

# 1. Load the raw Services PMI CSV file
df_pmi = pd.read_csv("Services PMI Data.csv")

# 2. Filter out rows containing footer notes or missing data values
df_pmi = df_pmi.dropna(subset=['Date', 'Services PMI']).copy()
df_pmi['Date'] = pd.to_datetime(df_pmi['Date'])

# 3. Dynamic Function to calculate the exact 3rd Business Day of the next month
def get_third_bday_next_month(dt):
    # Transition to the 1st day of the subsequent calendar month
    next_month = dt + pd.offsets.MonthBegin(1)
    
    curr = next_month
    bday_count = 0
    while bday_count < 3:
        if curr.weekday() <= 4:  # Monday to Friday are valid business days
            bday_count += 1
            if bday_count == 3:
                return curr
        curr += pd.Timedelta(days=1)
    return curr

# 4. Apply Calendar & Lag transformations
df_pmi['Release Date'] = df_pmi['Date'].apply(get_third_bday_next_month)

# Calculate Month End Date and snap back to Friday if it lands on a weekend
df_pmi['Month End Date'] = df_pmi['Date'] + pd.offsets.MonthEnd(0)
while any(df_pmi['Month End Date'].dt.weekday > 4):
    df_pmi.loc[df_pmi['Month End Date'].dt.weekday > 4, 'Month End Date'] -= pd.Timedelta(days=1)

# Format columns cleanly to standard strings
df_pmi['Month End Date'] = df_pmi['Month End Date'].dt.strftime('%Y-%m-%d')
df_pmi['Release Date'] = df_pmi['Release Date'].dt.strftime('%Y-%m-%d')

# 5. Isolate clean layout structure and export
final_pmi = df_pmi[['Month End Date', 'Release Date', 'Services PMI', 'Signal', 'Source']]
final_pmi.to_csv("Services_PMI_Cleaned.csv", index=False)

print("Services PMI Engine Complete. File saved.")
print(final_pmi.head(6))

Services PMI Engine Complete. File saved.
  Month End Date Release Date  Services PMI     Signal Source
0     2010-01-29   2010-02-03          59.0  Expansion     QZ
1     2010-02-26   2010-03-03          60.9  Expansion     QZ
2     2010-03-31   2010-04-05          58.1  Expansion     QZ
3     2010-04-30   2010-05-05          62.1  Expansion     QZ
4     2010-05-31   2010-06-03          58.2  Expansion     QZ
5     2010-06-30   2010-07-05          64.0  Expansion     QZ


In [13]:
#merging

import pandas as pd

df_cal = pd.read_csv("Monthly_Calendar_2010_2026.csv")
df_trade = pd.read_csv("Merchandise_Trade_Cleaned.csv")
df_gst = pd.read_csv("All_India_GST_Revenue_With_YoY.csv")
df_iip = pd.read_csv("IIP_Growth_Formatted_Spine.csv")
df_pmi = pd.read_csv("Services_PMI_Cleaned.csv")

df_cal['Release_Month'] = pd.to_datetime(df_cal['Last Day of Month']).dt.to_period('M')
df_trade['Release_Month'] = pd.to_datetime(df_trade['Release Date']).dt.to_period('M')
df_gst['Release_Month'] = pd.to_datetime(df_gst['Release Date']).dt.to_period('M')
df_iip['Release_Month'] = pd.to_datetime(df_iip['release_date']).dt.to_period('M')
df_pmi['Release_Month'] = pd.to_datetime(df_pmi['Release Date']).dt.to_period('M')

# 3. Filter target vectors — NOW INCLUDING RAW LEVEL COLUMNS for COVID base-effect fix
df_trade_sub = df_trade[['Release_Month', 'Exports_YoY_3MMA', 'Imports_NonOil_YoY_3MMA',
                          'Merchandise Exports', 'Non-Oil Imports']].drop_duplicates('Release_Month')
df_pmi_sub = df_pmi[['Release_Month', 'Services PMI']].drop_duplicates('Release_Month')
df_gst_sub = df_gst[['Release_Month', 'GST_YoY_Growth',
                      'Total GST Revenue (₹ Crore)']].drop_duplicates('Release_Month')
df_iip_sub = df_iip[['Release_Month', 'growth_3m_avg', 'IIP_Level']].rename(
    columns={'growth_3m_avg': 'IIP_Growth_3MMA'}).drop_duplicates('Release_Month')

merged = df_cal[['Last Day of Month', 'Release_Month']].copy()
merged = merged.merge(df_trade_sub, on='Release_Month', how='left')
merged = merged.merge(df_pmi_sub, on='Release_Month', how='left')
merged = merged.merge(df_gst_sub, on='Release_Month', how='left')
merged = merged.merge(df_iip_sub, on='Release_Month', how='left')

merged = merged.sort_values('Release_Month').reset_index(drop=True)

merged.to_csv("Merged_Base_Macro_Data.csv", index=False)
print("Step 1 Complete: 'Merged_Base_Macro_Data.csv' created.")

Step 1 Complete: 'Merged_Base_Macro_Data.csv' created.


In [14]:
import pandas as pd
import numpy as np

# 1. Load the merged file created in Step 1
df_merged = pd.read_csv("Merged_Base_Macro_Data.csv")
df_merged['Last Day of Month'] = pd.to_datetime(df_merged['Last Day of Month'])
df_merged = df_merged.sort_values('Last Day of Month').reset_index(drop=True)

# 1b. --- COVID BASE-EFFECT FIX: 2-Year Stacked CAGR ---
# For any month M where M-12 falls between Mar 2020 and Sep 2021,
# replace YoY with (Value_M / Value_M-24)^0.5 - 1, annualized over 2 years.

def apply_2y_stacked_cagr(df, level_col, yoy_col, dist_start='2020-03-01', dist_end='2021-09-01'):
    dist_start = pd.Timestamp(dist_start)
    dist_end = pd.Timestamp(dist_end)
    fixed = df[yoy_col].copy()

    for i in range(24, len(df)):
        month_minus_12 = df['Last Day of Month'].iloc[i] - pd.DateOffset(months=12)
        if dist_start <= month_minus_12 <= dist_end:
            val_m = df[level_col].iloc[i]
            val_m24 = df[level_col].iloc[i - 24]
            if pd.notna(val_m) and pd.notna(val_m24) and val_m24 != 0 and val_m / val_m24 > 0:
                fixed.iloc[i] = ((val_m / val_m24) ** 0.5 - 1) * 100  # *100 to match existing % scale
            else:
                fixed.iloc[i] = np.nan
    return fixed

level_map = {
    'Exports_YoY_3MMA': 'Merchandise Exports',
    'Imports_NonOil_YoY_3MMA': 'Non-Oil Imports',
    'GST_YoY_Growth': 'Total GST Revenue (₹ Crore)',
    'IIP_Growth_3MMA': 'IIP_Level',
}

for yoy_col, level_col in level_map.items():
    df_merged[yoy_col] = apply_2y_stacked_cagr(df_merged, level_col, yoy_col)

# 2. Define the Out-Of-Sample 36M Trailing Robust Z-Score with Winsorization
def rolling_robust_zscore_winsorized(series, window=36, min_periods=18):
    def calc_last_robust_z(window_slice):
        clean_slice = window_slice[~np.isnan(window_slice)]
        if len(clean_slice) < min_periods:
            return np.nan
        current_val = window_slice[-1]
        if np.isnan(current_val):
            return np.nan
        median = np.median(clean_slice)
        mad = np.median(np.abs(clean_slice - median))
        if mad == 0:
            std = np.std(clean_slice)
            z = (current_val - median) / std if std != 0 else 0.0
        else:
            z = (current_val - median) / (mad * 1.4826)
        return np.clip(z, -4.0, 4.0)
    return series.rolling(window=window, min_periods=1).apply(calc_last_robust_z, raw=True)

# 3. Apply rolling operations across all requested metrics
target_metrics = [
    'Exports_YoY_3MMA',
    'Imports_NonOil_YoY_3MMA',
    'Services PMI',
    'GST_YoY_Growth',
    'IIP_Growth_3MMA'
]

for metric in target_metrics:
    df_merged[f'{metric}_ZScore'] = rolling_robust_zscore_winsorized(df_merged[metric], window=36, min_periods=36) * -1

df_merged = df_merged.drop(columns=['Release_Month', 'Exports_YoY_3MMA', 'Imports_NonOil_YoY_3MMA',
                                     'Services PMI', 'GST_YoY_Growth', 'IIP_Growth_3MMA',
                                     'Merchandise Exports', 'Non-Oil Imports',
                                     'Total GST Revenue (₹ Crore)', 'IIP_Level'], errors='ignore')

# 4. Save the updated results back to CSV
df_merged.to_csv("Macro_Indicators_With_Robust_ZScores.csv", index=False)
print("Step 2 Complete: COVID base-effect corrected, Winsorized Z-Scores calculated and saved.")

Step 2 Complete: COVID base-effect corrected, Winsorized Z-Scores calculated and saved.


In [15]:
import pandas as pd
import numpy as np
import openpyxl

def apply_external_stress_formatting(input_csv, output_xlsx):
    # 1. Load the compiled data
    df = pd.read_csv(input_csv)
    
    # 2. Sort with newest dates on top to match standard spreadsheet layouts
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)
    df['Last Day of Month'] = df['Last Day of Month'].dt.strftime('%Y-%m-%d')
    
    # Identify target numeric columns to apply styling (exclude date columns)
    exclude_cols = ['Last Day of Month']
    z_cols = [col for col in df.columns if col not in exclude_cols]

    # 3. Define multi-tier styling logic
    # Positive values = Red (High Stress), Negative values = Green (Low Stress)
    def format_outliers_tiered(series):
        styles = []
        for val in series:
            if pd.isna(val):
                styles.append('')
            
            # --- POSITIVE DEVIATIONS (RED = HIGH EXTERNAL STRESS) ---
            elif val >= 2.0:
                styles.append('background-color: #ff9999; color: #660000; font-weight: bold;')
            elif val >= 1.0:
                styles.append('background-color: #ffe6e6; color: #990000;')
                
            # --- NEGATIVE DEVIATIONS (GREEN = LOW EXTERNAL STRESS) ---
            elif val <= -2.0:
                styles.append('background-color: #99ff99; color: #004d00; font-weight: bold;')
            elif val <= -1.0:
                styles.append('background-color: #e6ffe6; color: #006600;')
                
            # --- NORMAL/BALANCE RANGE ---
            else:
                styles.append('')
        return styles

    # 4. Apply styling and format to 4 decimal places
    styled_df = df.style.apply(format_outliers_tiered, subset=z_cols, axis=0)
    styled_df = styled_df.format({col: "{:.4f}" for col in z_cols})

    # 5. Export to Excel
    styled_df.to_excel(output_xlsx, index=False, engine='openpyxl')
    print(f"Success! Color-coded Excel sheet generated and saved to: {output_xlsx}")

# =========================================================================
# RUN THE PIPELINE: Put your actual file names inside the quotation marks below!
# =========================================================================

INPUT_CSV_FILE = "Macro_Indicators_With_Robust_ZScores.csv"
OUTPUT_EXCEL_FILE = "Growth_Scores.xlsx"

apply_external_stress_formatting(INPUT_CSV_FILE, OUTPUT_EXCEL_FILE)

Success! Color-coded Excel sheet generated and saved to: Growth_Scores.xlsx


In [16]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def map_to_5_point_scale(z_score):
    """
    Maps continuous Z-scores onto the updated 5-point scale: [-1.0, -0.5, 0.0, 0.5, 1.0].
    """
    if pd.isna(z_score): 
        return 0.0
    if z_score >= 1.4: 
        return 1.0     # Strongly Stressed / High Vulnerability
    elif z_score >= 1.0: 
        return 0.75     # Moderately Stressed
    elif z_score >= 0.55: 
        return 0.5     # Moderately Stressed
    elif z_score >= 0.25: 
        return 0.25     # Moderately Stressed
    elif z_score <= -1.4: 
        return -1.0    # Strongly Favorable / High Cushion
    elif z_score <= -1.0: 
        return -0.75    # Strongly Favorable / High Cushion
    elif z_score <= -0.55: 
        return -0.5    # Strongly Favorable / High Cushion
    elif z_score <= -0.25: 
        return -0.25    # Moderately Favorable
    else: 
        return 0.0     # Neutral Anchor    

def classify_net_to_7_point_regime(net_score):
    """
    True 7-Point Regime Scale: [-3, -2, -1, 0, 1, 2, 3]
    Stabilized with a wider neutral buffer zone to absorb micro-swings.
    """
    abs_score = abs(net_score)
    sign = np.sign(net_score)
    
    if abs_score >= 0.6:
        regime = 3.0    # Extreme Macro Tail Risk
    elif abs_score >= 0.35:
        regime = 2.0    # Strong Expansion / Contraction
    elif abs_score >= 0.15:
        regime = 1.0    # Mild Orderly Trend
    else:
        regime = 0.0    # Structural Neutral Anchor
        
    return regime * sign

def apply_asymmetric_lookback_filter(regimes):
    """
    Advanced Chronological Timing Engine.
    Implements a 'Fast-Attack, Slow-Decay' asymmetric gate.
    - Fast Attack: Maximum tail shocks (+3 or -3) bypass the window and print instantly.
    - Slow Decay: Protects the portfolio from immediate structural whipsaws by 
      forcing a controlled step-down when exiting a severe crisis state.
    """
    confirmed = []
    current_confirmed = 0.0
    
    for i in range(len(regimes)):
        flash = regimes[i]
        
        if i < 2:
            current_confirmed = flash if pd.notna(flash) else 0.0
            confirmed.append(current_confirmed)
            continue
            
        # 1. FAST ATTACK GATE
        # If the current month encounters a severe systemic fracture, 
        # breach the lookback filter immediately to flag the risk.
        if flash == 3.0 or flash == -3.0:
            current_confirmed = flash
            confirmed.append(current_confirmed)
            continue
            
        # 2. STANDARD FILTER PROCESSING (For normal market conditions)
        window = [regimes[i], regimes[i-1], regimes[i-2]]
        window = [v for v in window if pd.notna(v)]
        
        if len(window) == 0:
            confirmed.append(current_confirmed)
            continue
            
        counts = pd.Series(window).value_counts()
        highest_frequency = counts.iloc[0]
        most_frequent_value = counts.index[0]
        
        if highest_frequency >= 2:
            proposed_state = most_frequent_value
        else:
            proposed_state = float(np.median(window))
            
        # 3. SLOW DECAY GATE (Hysteresis Loop)
        # If the portfolio was just in a Severe Crisis (3.0), do not allow the system 
        # to jump straight back into aggressive risk-on environments in 30 days. 
        # Force a mandatory safety cushion floor of Mild Headwinds (1.0).
        if current_confirmed == 3.0 and proposed_state < 1.0:
            current_confirmed = 1.0  # Safe step-down floor
        elif current_confirmed == -3.0 and proposed_state > -1.0:
            current_confirmed = -1.0 # Safe step-up ceiling
        else:
            current_confirmed = proposed_state
            
        confirmed.append(current_confirmed)
        
    return confirmed

def process_growth_pillar(input_excel_path, output_excel_path):
    # 1. Load data natively from Excel
    df = pd.read_excel(input_excel_path)
    
    # Force chronological alignment (Oldest history to Newest) for accurate lookback looping
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month').reset_index(drop=True)
    
    results = []
    target_weights = {'pmi': 0.35, 'gst': 0.15, 'imp': 0.05, 'iip': 0.3, 'exp': 0.15}
    
    for idx, row in df.iterrows():
        pmi_z = row['Services PMI_ZScore']
        imp_z = row['Imports_NonOil_YoY_3MMA_ZScore']
        gst_z = row['GST_YoY_Growth_ZScore']
        iip_z = row['IIP_Growth_3MMA_ZScore']
        exp_z = row['Exports_YoY_3MMA_ZScore']
        
        pmi_f = map_to_5_point_scale(pmi_z) if pd.notna(pmi_z) else None
        imp_f = map_to_5_point_scale(imp_z) if pd.notna(imp_z) else None
        gst_f = map_to_5_point_scale(gst_z) if pd.notna(gst_z) else None
        iip_f = map_to_5_point_scale(iip_z) if pd.notna(iip_z) else None
        exp_f = map_to_5_point_scale(exp_z) if pd.notna(exp_z) else None
        
        weighted_sum = 0.0
        total_active_weight = 0.0
        
        if pmi_f is not None: weighted_sum += pmi_f * target_weights['pmi']; total_active_weight += target_weights['pmi']
        if gst_f is not None: weighted_sum += gst_f * target_weights['gst']; total_active_weight += target_weights['gst']
        if imp_f is not None: weighted_sum += imp_f * target_weights['imp']; total_active_weight += target_weights['imp']
        if iip_f is not None: weighted_sum += iip_f * target_weights['iip']; total_active_weight += target_weights['iip']
        if exp_f is not None: weighted_sum += exp_f * target_weights['exp']; total_active_weight += target_weights['exp']
            
        if total_active_weight == 0.0:
            results.append({'Base_Net_Score': np.nan, 'Adjusted_Net_Score': np.nan, 'Flash_Regime': np.nan})
            continue
            
        base_net_score = weighted_sum / total_active_weight
        
        # Two-Speed Economy Dislocation Detection
        dom_sum, dom_w = 0.0, 0.0
        if pmi_f is not None: dom_sum += pmi_f * target_weights['pmi']; dom_w += target_weights['pmi']
        if gst_f is not None: dom_sum += gst_f * target_weights['gst']; dom_w += target_weights['gst']
        if imp_f is not None: dom_sum += imp_f * target_weights['imp']; dom_w += target_weights['imp']
        
        domestic_core = dom_sum / dom_w if dom_w > 0 else 0.0
        external_core = exp_f if exp_f is not None else 0.0
        
        if domestic_core * external_core < 0:
            divergence_magnitude = abs(domestic_core - external_core)
            if domestic_core > 0 and external_core < 0:
                dislocation_modifier = 0.15 * (divergence_magnitude / 2.0)
            else:
                dislocation_modifier = -0.15 * (divergence_magnitude / 2.0)
            adjusted_net_score = base_net_score + dislocation_modifier
        else:
            adjusted_net_score = base_net_score
            
        final_net_score = max(-1.0, min(1.0, adjusted_net_score))
        flash_regime = classify_net_to_7_point_regime(final_net_score)
        
        results.append({
            'Base_Net_Score': base_net_score,
            'Adjusted_Net_Score': adjusted_net_score,
            'Flash_Regime': flash_regime
        })

    # Combine calculations into the dataframe
    res_df = pd.DataFrame(results)
    output_df = pd.concat([df, res_df], axis=1)
    
    # 4. RUN CHRONOLOGICAL LOOKBACK FILTER
    output_df['Growth_Confirmed_Regime'] = apply_asymmetric_lookback_filter(output_df['Flash_Regime'].tolist())
    
    # Drop the intermediate flash regime column to keep only the final confirmed column
    output_df = output_df.drop(columns=['Flash_Regime'])
    
    # PRESENTATION FLIP: Reverse sort table so 2026 sits cleanly at the top
    output_df = output_df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)
    
    # Export parsed dataframe to Excel to begin structural styling
    output_df.to_excel(output_excel_path, index=False)
    
    # 5. SHEET STYLING & PROFESSIONAL POLISHING (Openpyxl Engine)
    wb = openpyxl.load_workbook(output_excel_path)
    ws = wb.active
    ws.title = "Growth Model Analysis"
    
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )
    
    # Format Headers
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        
    # Format Data Rows
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.border = thin_border
            if cell.column == 1:
                if isinstance(cell.value, (pd.Timestamp, np.datetime64)):
                    cell.number_format = 'yyyy-mm-dd'
                cell.alignment = Alignment(horizontal="center") 
            else:
                cell.alignment = Alignment(horizontal="right")   
                if cell.value is not None and isinstance(cell.value, (int, float)):
                    cell.number_format = '0.00'                  

    # Auto-adjust column widths dynamically to prevent cell text clipping
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = openpyxl.utils.get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)
        
    # Freeze header panel and dates column in place during scroll navigation
    ws.freeze_panes = 'B2'
    
    wb.save(output_excel_path)
    print(f"Process complete! Output successfully saved to: {output_excel_path}")

# Run the full suite using your workbook
process_growth_pillar("Growth_Scores.xlsx", "Growth_Pillar_Model_Outputs.xlsx")

Process complete! Output successfully saved to: Growth_Pillar_Model_Outputs.xlsx
